In [ ]:
# Databricks notebook source
# 07_union_all

import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from pyspark.sql.types import DateType, DoubleType, StringType, StructField, StructType

CATALOG = "decide_catalog"
SCHEMA = "decide_schema"
VOLUME_PATH = "/Volumes/decide_catalog/decide_schema/decide_volume"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")


def should_run_pipeline():
    try:
        value = dbutils.jobs.taskValues.get(taskKey="00_check_source_changes", key="should_run", default="true")
        return str(value).lower() == "true"
    except Exception:
        return True


def source_path(file_name):
    return f"{VOLUME_PATH}/{file_name}"


def sha256_hash(value):
    if pd.isna(value):
        return None
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def month_start(values):
    return pd.to_datetime(values, errors="coerce").dt.to_period("M").dt.start_time.dt.date


def week_start(values, week_start="sunday"):
    dates = pd.to_datetime(values, errors="coerce")
    freq = "W-SAT" if week_start == "sunday" else "W-SUN"
    return dates.dt.to_period(freq).dt.start_time.dt.date


def max_with_na(series):
    values = pd.to_numeric(series, errors="coerce")
    if values.notna().any():
        return values.max(skipna=True)
    return np.nan


def write_delta(pdf, table_name):
    output_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Date",
        "Date_month",
        "Date_week",
        "Pathogen",
        "Result",
    ]
    pdf = pdf.reindex(columns=output_cols).copy()

    string_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Pathogen",
    ]
    for col in string_cols:
        pdf[col] = pdf[col].where(pd.notna(pdf[col]), None).astype(object)
    for col in ["Date", "Date_month", "Date_week"]:
        pdf[col] = pd.to_datetime(pdf[col], errors="coerce").dt.date
    pdf["Result"] = pd.to_numeric(pdf["Result"], errors="coerce")

    schema = StructType(
        [
            StructField("Lab_reference", StringType(), True),
            StructField("Country", StringType(), True),
            StructField("Breed", StringType(), True),
            StructField("Province", StringType(), True),
            StructField("Farm_ID", StringType(), True),
            StructField("Diagnostic_test", StringType(), True),
            StructField("Sample_type", StringType(), True),
            StructField("Samplenumber", StringType(), True),
            StructField("Date", DateType(), True),
            StructField("Date_month", DateType(), True),
            StructField("Date_week", DateType(), True),
            StructField("Pathogen", StringType(), True),
            StructField("Result", DoubleType(), True),
        ]
    )
    sdf = spark.createDataFrame(pdf, schema=schema)
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    (
        sdf.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )
    print(f"Wrote {sdf.count()} rows to {full_name}")

from functools import reduce
from pyspark.sql import functions as F

if not should_run_pipeline():
    dbutils.notebook.exit("No source changes detected; skipping union")

table_names = [
    "barometer_arsia",
    "barometer_dgz",
    "barometer_gd",
    "barometer_ireland",
    "barometer_pathosense",
    "barometer_vigigrip",
]

frames = [spark.table(f"{CATALOG}.{SCHEMA}.{name}") for name in table_names]
combined = reduce(lambda left, right: left.unionByName(right), frames)
combined = (
    combined
    .withColumn("Month", F.month("Date"))
    .withColumn("Year", F.year("Date"))
    .withColumnRenamed("Lab_reference", "LabReference")
    .withColumnRenamed("Farm_ID", "FarmIdentifier")
    .withColumnRenamed("Diagnostic_test", "DiagnosticTest")
    .withColumnRenamed("Sample_type", "SampleType")
)

(
    combined.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.barometer_combined")
)
print(f"Wrote {combined.count()} rows to {CATALOG}.{SCHEMA}.barometer_combined")

# Record the source metadata only after all cleaning tasks and the union complete successfully.
listed_files = {item.name: item for item in dbutils.fs.ls(VOLUME_PATH)}
source_files = [
    "DECIDE_COMPIL_DATA__202501272012.xlsx",
    "MTA_DECIDE_Ugent2025.xlsx",
    "DECIDE_MTA_UGENT_BAC_AERO_14nov2022.xlsx",
    "DECIDE_MTA_UGENTBAC_MYCO_14nov2022.xlsx",
    "250808_data_RGD_DECIDE.xlsx",
    "Jade_2021_Final_Anonymised_data_Only_2023-04-20.v2.xlsx",
    "Jade_2022_Final_Anonymised_data_Only_2023-04-21.xlsx",
    "Final_Anonymised_data_Only_2023_2025-03-04.xlsx",
    "AllBovineRespiratory_NegativesIncluded.csv",
    "DECIDE_final_version.xlsx",
    "gd_labresults.csv",
]
state_rows = [
    {
        "file_name": name,
        "file_path": listed_files[name].path,
        "file_size": int(listed_files[name].size),
        "modification_time_ms": int(listed_files[name].modificationTime),
        "checked_at_utc": datetime.now(timezone.utc),
    }
    for name in source_files
]
state_df = spark.createDataFrame(state_rows)
(
    state_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.source_file_state")
)
print(f"Updated {CATALOG}.{SCHEMA}.source_file_state")
